 # 加载全A和指数合并的数据

In [ ]:
# 1. 导入必要的库
import qlib
from qlib.constant import REG_CN
import logging
import os
import pandas as pd
import numpy as np
from qlib.data import D
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')


#设置参数
data_category='allAShare_with_IndexValue'

# 初始化qlib
current_directory = os.getcwd()
print("当前工作目录:", current_directory)
provider_url = os.path.abspath(f"./.qlib/qlib_data/{data_category}/")
qlib.init(provider_uri=provider_url, region=REG_CN, logging_level=logging.INFO)


In [ ]:
# 指数名称到代码的映射字典
INDEX_NAME_TO_CODE = {
    'shangzheng50': '000016.SH',      # 上证50
    'hushen300': '000300.SH',         # 沪深300
    'zhongzheng500': '000905.SH',     # 中证500
    'zhongzheng1000': '000852.SH',    # 中证1000
}

def extract_index_code_from_filename(file_path):
    """
    从文件路径中提取指数名称并返回对应的指数代码
    
    Args:
        file_path (str): 包含指数名称的文件路径，如 '.qlib/indicator_data/zhongzheng1000_PB.csv'
    
    Returns:
        str: 对应的指数代码，如 '000852.SH'，如果未找到则返回 None
    """
    import re
    
    # 提取文件名（不包含路径和扩展名）
    filename = os.path.basename(file_path)
    filename_without_ext = os.path.splitext(filename)[0]
    
    # 使用正则表达式提取指数名称（在第一个下划线之前的部分）
    match = re.match(r'^([a-zA-Z]+[0-9]+)', filename_without_ext)
    if match:
        index_name = match.group(1)
        return INDEX_NAME_TO_CODE.get(index_name)
    
    return None

In [ ]:
# 回测时间范围确定
start_time = '2015-01-05'
end_time = '2024-12-31'

# 通过csv创建信号值

In [ ]:

factor_csv_file_path = '.qlib/indicator_data/hushen300_Top3_return_log.csv'

from pathlib import Path
factor_name_pathlib = Path(factor_csv_file_path).stem
print(f"文件名(pathlib): {factor_name_pathlib}")
your_signal_df = pd.read_csv(factor_csv_file_path)
print(your_signal_df.info())
print(your_signal_df.head())

# 因子预处理

In [ ]:
# 删除多余列（如从 CSV 读入产生）
if 'Unnamed: 0' in your_signal_df.columns:
    your_signal_df = your_signal_df.drop(columns=['Unnamed: 0'])

# 统一列名
df = your_signal_df.rename(columns={'code': 'instrument', 'pb': 'pb', 'abnormal_gross_margin': 'abnormal_gross_margin', 'change_percent_20d': 'change_percent_20d'})

# 类型转换
df['datetime'] = pd.to_datetime(df['datetime'])
df['Top3_return_log'] = pd.to_numeric(df['Top3_return_log'], errors='coerce').astype('float32')

# 设为 MultiIndex，并排序
multi_df = df.set_index(['instrument', 'datetime']).sort_index()

# 查看结果
print(multi_df.info())
print(multi_df.head())

In [ ]:
# 将DataFrame转换为Series并清理数据
signal_series =multi_df['Top3_return_log'].copy()

# 数据清理：移除无效值
print(f"原始信号数据形状: {signal_series.shape}")
print(f"原始数据中NaN数量: {signal_series.isna().sum()}")

# 填充或移除NaN值
signal_series = signal_series.dropna()
print(f"清理后信号数据形状: {signal_series.shape}")

# 创建信号对象 - 使用正确的方法
from qlib.backtest.signal import create_signal_from

# 根据源码只接受一个参数
signal_obj = create_signal_from(signal_series)

print("信号对象创建完成")
print(f"信号对象类型: {type(signal_obj)}")



# 策略定义

In [ ]:
# 3. 定义等权策略类(月初第一个交易日调仓)
from qlib.contrib.strategy import WeightStrategyBase
from qlib.backtest import backtest, executor
import pandas as pd
from qlib.data import D
class EqualWeightStrategy(WeightStrategyBase):
    """
    等权策略：选择前topk只股票等权重分配
    """
    def __init__(self, start_percent=0,end_percent=20,topk=20,rebalance_day=20, **kwargs):
        super().__init__(**kwargs)
        self.topk = topk
        self.start_percent=start_percent
        self.end_percent=end_percent
        self.rebalance_day=rebalance_day
        self.trade_day_count=0
        self.pending_sell=[]
        self.coalesced_stocks=[]


    def is_first_trading_day_of_month(self,trade_start_time, target_months=[5, 8, 11]):
        """
        判断是否为目标月份的第一个交易日
        
        参数:
        - trade_start_time: 交易开始时间
        - target_months: 目标月份列表
        
        返回:
        - bool: 是否为目标月份的第一个交易日
        """
        current_month = trade_start_time.month
        current_year = trade_start_time.year
        
        print(f"🔍 调仓日判断 - 日期: {trade_start_time}, 月份: {current_month}, 目标月份: {target_months}")
        
        # 检查是否为目标月份
        if current_month not in target_months:
            print(f"❌ 月份 {current_month} 不在目标月份中")
            return False
        
        print(f"✅ 月份 {current_month} 在目标月份中")
        
        # 获取该月的第一个交易日
        start_date = pd.Timestamp(f"{current_year}-{current_month:02d}-01")
        end_date = start_date + pd.DateOffset(months=1) - pd.DateOffset(days=1)
        
        print(f"📅 查询交易日历范围: {start_date} 到 {end_date}")
        
        # 使用Qlib的交易日历获取该月第一个交易日
        trading_days = D.calendar(start_time=start_date, end_time=end_date)
        print(f"📊 交易日数量: {len(trading_days)}")
        
        if len(trading_days) > 0:
            first_trading_day = trading_days[0]
            print(f"📅 第一个交易日: {first_trading_day}")
            print(f"🔍 比较: {trade_start_time.date()} == {first_trading_day.date()}")
            # 比较日期（只比较日期部分，忽略时间）
            result = trade_start_time.date() == first_trading_day.date()
            print(f"🎯 调仓日判断结果: {result}")
            return result
        
        print("❌ 没有交易日，返回False")
        return False

    def generate_target_weight_position(self, score, current, trade_start_time, trade_end_time):
        """
        生成目标权重仓位
        
        参数:
            score: pd.Series，索引为股票代码，值为预测分数
            current: 当前持仓对象
            trade_start_time: 交易开始时间
            trade_end_time: 交易结束时间
            
        返回:
            dict，键为股票代码，值为目标权重
        """

        
        if score is None:
            print("score为None，返回空权重")
            return 
        
        self.trade_day_count+=1
        
        # 添加详细的持仓变化跟踪

        # 获取当前持仓
        current_holdings = current.position
        current_position_weights = {}
        for stock_code, stock_info in current_holdings.items():
            if stock_code not in ['cash', 'now_account_value']:
                current_position_weights[stock_code] = stock_info.get('weight', 0)
       

        print(f"🔍 第{self.trade_day_count}个交易日 {trade_start_time}")
        print(f"📊 当前持仓股票数: {len(current_position_weights)}")
        print(f"📋 待卖出列表: {self.pending_sell}")
        print(f"📈 当前持仓权重: {current_position_weights}")
        print(f"📈 当前信号值: {score.sort_values(ascending=True).to_dict()}")
       

        # 1 检查是否合并或者调离指数
        if not self.is_first_trading_day_of_month(trade_start_time):
    
            # 检查score中是否有-99999和-99998值
            invalid_stocks_99999 = score[score == -99999].index.tolist()
            invalid_stocks_99998 = score[score == -99998].index.tolist()
            
            # 处理-99999的股票（添加到待卖出列表）
            if len(invalid_stocks_99999) > 0:
                print(f"发现无效信号股票(-99999),出现调离指数: {invalid_stocks_99999}")
                for stock_id in invalid_stocks_99999:
                    if stock_id not in self.pending_sell and stock_id  in current_position_weights:
                        self.pending_sell.append(stock_id)
                        print(f"已将 {stock_id} 添加到待卖出列表")
            
            # 处理-99998的股票（添加到coalesced_stocks）
            if len(invalid_stocks_99998) > 0:
                print(f"发现合并股票(-99998): {invalid_stocks_99998}")
                for stock_id in invalid_stocks_99998:
                    
                    if stock_id not in self.coalesced_stocks:
                        self.coalesced_stocks.append(stock_id)
                        print(f"已将 {stock_id} 添加到coalesced_stocks列表")

                    if stock_id not in self.pending_sell and stock_id  in current_position_weights:
                        self.pending_sell.append(stock_id)
                        print(f"已将 {stock_id} 添加到待卖出列表")    
                            


        # 2 处理待卖出列表
        if len(self.pending_sell) !=0 and not self.is_first_trading_day_of_month(trade_start_time):

            print(f"当前时间为{trade_start_time}，开始处理待卖出列表{self.pending_sell}")
            tradable_pending_sell_stock_id=[]
            tradable_pending_sell_dict={}

            for stock_id in self.pending_sell:
                if self.trade_exchange.is_stock_tradable(
                    stock_id=stock_id,
                    start_time=trade_start_time,
                    end_time=trade_end_time
                ):
                    tradable_pending_sell_stock_id.append(stock_id)
            print(f"可交易的的待卖出列表中的股票: {tradable_pending_sell_stock_id}")
            for stock_id in tradable_pending_sell_stock_id:
                self.pending_sell.remove(stock_id)
                tradable_pending_sell_dict[stock_id]=0
                current_position_weights[stock_id]=0
            
            print(f"可卖出的待卖出列表的权重字典{tradable_pending_sell_dict}")
            if len(tradable_pending_sell_dict) > 0:
                return current_position_weights

            else:
                
                print(f"没有可卖出的待卖出股票，继续执行后续逻辑")  
                return
        # 3 调仓日重新选择股票组合
        if self.is_first_trading_day_of_month(trade_start_time):

            print(f"🔄 调仓日 - 重新选择股票组合")
            print(f"交易开始时间{trade_start_time}，交易结束时间{trade_end_time}")
            
            # 现在current_weights就包含了当前所有持仓及其权重
            print(f"当前持仓权重: {current_position_weights}")
            print(f'当前持仓股票总数：{len(current_position_weights)}')

            # 选择前 topk 只股票（只从有效信号中选择）
            if len(score) > 0:
                
                # 排除在coalesced_stocks中的股票
                if hasattr(self, 'coalesced_stocks') and len(self.coalesced_stocks) > 0:
                    valid_score = score[(~score.index.isin(self.coalesced_stocks)) & (~score.isin([-99999, -99998]))]
                    print(f"排除了coalesced_stocks中的股票: {self.coalesced_stocks}")
                    print(f"排除后剩余有效信号数量: {len(valid_score)}")
                
                if len(valid_score) > 0:
                    # 确保有足够的股票
                    # actual_topk = min(self.topk, len(valid_score))
                    # sorted_score = valid_score.sort_values(ascending=True)
                    # selected_stocks = sorted_score.head(actual_topk).index
                    # print(f"实际选择股票数量: {actual_topk}")
                    # print(f"当天有效的的股票因子值: {valid_score.sort_values(ascending=True).to_dict()}")
                    # print(f"选择的股票的因子值: {sorted_score.sort_values(ascending=True).to_dict()}")
                    # 选择start_percent和end_percent之间的股票
                    total_stocks = len(valid_score)
                    start_index = int(total_stocks * self.start_percent)
                    end_index = int(total_stocks * self.end_percent)
                    actual_count = end_index - start_index
                    if actual_count > 0:
                        sorted_score = valid_score.sort_values(ascending=True)
                        # 按百分比选择股票
                        selected_stocks = sorted_score.iloc[start_index:end_index].index
                        print(f"总股票数: {total_stocks}, 选择范围: {start_index}-{end_index}, 实际选择股票数量: {actual_count}")
                        print(f"当天有效的股票因子值: {valid_score.sort_values(ascending=True).to_dict()}")
                        print(f"选择的股票因子值: {sorted_score.iloc[start_index:end_index].to_dict()}")
                    else:
                        print("按百分比计算后没有股票可选")
                        selected_stocks = pd.Index([])
                else:
                    print("所有信号都是无效值")
                    selected_stocks = pd.Index([])
            else:
                print("score为空")
                selected_stocks = pd.Index([])

            print(f"选择的股票数量: {len(selected_stocks)}")
          
            if len(selected_stocks) > 0:
                print(f"选择的股票: {selected_stocks.tolist()}...")  # 显示前5只
            
            # 为每只股票分配相等的权重，并添加归一化处理
            if len(selected_stocks) > 0:
                # 检查可交易性并筛选可交易股票
                current_tradable_stocks = []
                current_non_tradable_stocks = []

                for stock_id in current_position_weights.keys():
                    if  self.trade_exchange.is_stock_tradable(
                        stock_id=stock_id,
                        start_time=trade_start_time,
                        end_time=trade_end_time
                    ):
                        current_tradable_stocks.append(stock_id)
                    else:
                        current_non_tradable_stocks.append(stock_id)


                print(f"当前持仓可交易股票数量: {len(current_tradable_stocks)}")
                print(f"当前持仓可交易股票: {current_tradable_stocks}")

                print(f"当前持仓不可交易股票数量: {len(current_non_tradable_stocks)}")
                print(f"当前持仓不可交易股票: {current_non_tradable_stocks}")

                new_group_tradable_stocks = []
                for stock_id in selected_stocks:
                    if self.trade_exchange.is_stock_tradable(
                        stock_id=stock_id,
                        start_time=trade_start_time,
                        end_time=trade_end_time
                    ):
                        new_group_tradable_stocks.append(stock_id)
                        
                print(f"新分组可交易股票数量: {len(new_group_tradable_stocks)}")
                print(f"新分组可交易股票: {new_group_tradable_stocks}")

                if len(new_group_tradable_stocks) > 0:
                    # 初始化目标权重字典
                    target_weight_dict = {}

                    # 计算当前持仓无法交易股票的总权重
                    non_tradable_weight_sum = 0
                    for stock_id in current_non_tradable_stocks:
                        non_tradable_weight_sum += current_position_weights[stock_id]
                        #将当前持仓无法交易的股票迁移至目标权重字典(是否无需迁移 因为target_weight_dict中不包含就不会操作)
                        target_weight_dict[stock_id] = current_position_weights[stock_id]
                    print(f"当前持仓无法交易股票总权重: {non_tradable_weight_sum:.4f}")
                    
                    # 找出不在当前分组但在持仓中且可交易的股票，将其权重设置为0
                    current_position_tradable_stocks = set(current_tradable_stocks)
                    current_position_non_tradable_stocks = set(current_non_tradable_stocks)
                    current_selection_stocks = set(selected_stocks)
                    stocks_to_zero = current_position_tradable_stocks - current_selection_stocks

                    #找出不在当前分组但在持仓中且不可交易的股票，将其加入待卖出列表
                    stock_to_sell=current_position_non_tradable_stocks - current_selection_stocks
                    for stock_id in stock_to_sell:
                        if stock_id not in self.pending_sell:
                            self.pending_sell.append(stock_id)


                    print(f"当前需权重归零的股票数量: {len(stocks_to_zero)}")

                    print(f"当前需权重归零的股票: {list(stocks_to_zero)}")
                    for stock_id in stocks_to_zero:
                        target_weight_dict[stock_id] = 0
                    
                    # 计算剩余可交易权重
                    available_weight = 1.0 - non_tradable_weight_sum
                    print(f"剩余可交易权重: {available_weight:.4f}")
                    
                    # 对可交易股票进行等权分配
                    if available_weight > 0 and len(new_group_tradable_stocks) > 0:
                        equal_weight = available_weight / len(new_group_tradable_stocks)
                        for stock_id in new_group_tradable_stocks:
                            target_weight_dict[stock_id] = equal_weight
                    
                    print(f"目标权重分配: {target_weight_dict}")
                    print(f"目标权重字典总权重: {sum(target_weight_dict.values()):.4f}")

                    print("\n")

                    return target_weight_dict
        else:
            print(f"⏸️ 非调仓日且无待卖出股票，保持当前持仓")
            return 


In [ ]:
# 4. 构建回测配置
# 策略配置 - 关键：使用正确的信号对象
strategy_config = {
    "topk": 20,                    # 选择前20只股票
    "start_percent":0,
    "end_percent":0.2,
    "signal": signal_obj,          # 【重要】使用信号对象而不是原始数据
    "rebalance_day": 20,
    "risk_degree": 1,
}

# 执行器配置
executor_config = {
    "time_per_step": "day",
    "generate_portfolio_metrics": True,
}

# 回测参数配置
backtest_config = {
    "start_time": start_time,
    "end_time": end_time,
    "account": 100000000,          # 初始资金 1亿
    "benchmark": extract_index_code_from_filename(factor_csv_file_path),      # 基准指数
    "exchange_kwargs": {
        "limit_threshold": ('$limit_up', '$limit_down'),   # 涨跌停限制
        "deal_price": "close",     # 以收盘价交易
        "open_cost": 0.0005,       # 开仓手续费
        "close_cost": 0.0005,      # 平仓手续费
        "min_cost": 50,             # 最低手续费
    },
}

print("=== 运行回测 ===")

# 重新运行回测
try:
    # 实例化策略和执行器
    strategy_obj = EqualWeightStrategy(**strategy_config)
    executor_obj = executor.SimulatorExecutor(**executor_config)

    print(f"策略对象创建完成: {type(strategy_obj)}")
    print(f"执行器对象创建完成: {type(executor_obj)}")
    print(f"信号对象类型: {type(signal_obj)}")

    # 执行回测
    print("开始执行回测...")
    portfolio_metric_dict, indicator_dict = backtest(
        executor=executor_obj,
        strategy=strategy_obj,
        **backtest_config
    )

    print("回测完成！")
    print(f"可用的回测频率: {list(portfolio_metric_dict.keys())}")
    
    # 检查回测结果
    if '1day' in portfolio_metric_dict:
        report_df, positions_dict = portfolio_metric_dict['1day']
        print("=" * 50)
        print("回测结果概览:")
        print(f"回测天数: {len(report_df)}")
        print(f"持仓记录数: {len(positions_dict)}")
        
        # 检查是否有交易
        total_turnover = report_df['total_turnover'].mean()
        print(f"平均换手率: {total_turnover:.2f}")
        
        if total_turnover > 0:
            print("✓ 回测成功！有交易发生")
        else:
            print("✗ 回测失败！仍然没有交易")
            
except Exception as e:
    print(f"回测出错: {e}")
    import traceback
    print(traceback.format_exc())


# 结果输出

print("7. Position对象结构总结:")
print("   Position对象包含以下主要部分:")
print("   ├── _settle_type: 结算类型标识")
print("   └── position: 核心持仓字典，包含:")
print("       ├── 'cash': 现金余额")
print("       ├── 'now_account_value': 当前账户总价值")
print("       ├── 'cash_delay': 延迟结算的现金（如果有）")
print("       └── 股票代码: 每只股票的详细信息")
print("           ├── 'amount': 持股数量")
print("           ├── 'price': 股票价格")
print("           ├── 'weight': 权重")
print("           └── 'count_day': 持股天数")

In [ ]:
# 定义目标文件夹路径
folder_path = f'./logs/{factor_name_pathlib}'
# 核心步骤：创建文件夹
os.makedirs(folder_path, exist_ok=True)  


In [ ]:

report_df, positions_dict = portfolio_metric_dict['1day']
 

In [ ]:

# 获取第一个日期作为示例
from random import sample
from turtle import position


sample_date = list(positions_dict.keys())[1]
print(positions_dict.keys())
print(type(positions_dict[sample_date]))
print(positions_dict[sample_date].position.keys())

In [ ]:
import datetime
# 查看绩效报告的列名，确保要用的列都存在
print(report_df.columns.tolist())

# 计算并添加一些常用指标，例如累计收益率
report_df['cumulative_return'] = (1 + report_df['return']).cumprod() - 1
report_df['bench_cumulative_return'] = (1 + report_df['bench']).cumprod() - 1

# 选择关心的列输出
key_metrics_df = report_df[['account', 'return', 'cumulative_return', 'bench', 'bench_cumulative_return', 'turnover', 'total_cost']]

now_time=datetime.datetime.now()

print(key_metrics_df.info()) # 查看最后10天的情况
# 可以将 DataFrame 保存到CSV文件
key_metrics_df.to_csv(f'./logs/{factor_name_pathlib}/{factor_name_pathlib}_回测绩效报告.csv', encoding='utf-8-sig')

In [ ]:
# 分成两张表的持仓信息处理代码
import pandas as pd
from datetime import datetime


from qlib import data

def process_holdings_data_separated(positions_dict, output_dir=f'logs/{factor_name_pathlib}'):
    """
    将持仓数据分成两张表：
    1. 每日账户汇总表（现金和账户价值）
    2. 每日股票持仓表（具体持仓信息）
    """
    if not positions_dict:
        print("持仓数据为空")
        return pd.DataFrame(), pd.DataFrame()
    
    # 生成时间戳
    formatted_time = datetime.now().strftime('%Y%m%d_%H%M')
    
    # 存储每日账户汇总信息
    daily_account_list = []
    # 存储每日股票持仓信息
    daily_holdings_list = []
    
    # 按日期排序，确保时间顺序
    sorted_dates = sorted(positions_dict.keys())
    
    for i, date_str in enumerate(sorted_dates):
        position_obj = positions_dict[date_str]
        
        try:
            # 安全地获取持仓信息
            if hasattr(position_obj, 'position'):
                holdings_info = position_obj.position
            elif isinstance(position_obj, dict):
                holdings_info = position_obj
            else:
                print(f"未知的持仓对象类型: {type(position_obj)}")
                continue
            
            # 安全地提取现金和总资产，使用get方法避免KeyError
            cash = holdings_info.get('cash', 0)
            account_value = holdings_info.get('now_account_value', 0)
            
            # 如果当前日期没有现金信息，尝试从前一个交易日获取
            if cash == 0 and i > 0:
                prev_date = sorted_dates[i-1]
                prev_position = positions_dict[prev_date]
                if hasattr(prev_position, 'position'):
                    prev_holdings = prev_position.position
                    cash = prev_holdings.get('cash', 0)
                    print(f"从 {prev_date} 获取现金信息: {cash}")
            
            # 创建每日账户汇总记录（表1：账户汇总）
            daily_account_data = {
                'date': pd.to_datetime(date_str),
                'cash': cash,
                'account_value': account_value,
                'stock_count': 0,  # 持股数量
                'total_position_value': 0,  # 总持仓价值
                'cash_ratio': 0,  # 现金比例
                'position_ratio': 0  # 持仓比例
            }
            
            # 遍历该日期持有的每一只股票，统计信息
            stock_count = 0
            total_position_value = 0
            
            for stock_code, stock_info in holdings_info.items():
                if (stock_code not in ['cash', 'now_account_value'] and 
                    isinstance(stock_info, dict) and 
                    stock_info.get('amount', 0) > 0):  # 只处理有持仓的股票
                    
                    # 创建股票持仓记录（表2：股票持仓）
                    stock_data = {
                        'date': pd.to_datetime(date_str),
                        'stock_code': stock_code,
                        'amount': stock_info.get('amount', 0),
                        'price': stock_info.get('price', 0),
                        'weight': stock_info.get('weight', 0),
                        'count_day': stock_info.get('count_day', 0),
                        'position_value': 0
                    }
                    
                    # 计算持仓价值
                    amount = stock_data.get('amount', 0)
                    price = stock_data.get('price', 0)
                    position_value = amount * price
                    stock_data['position_value'] = position_value
                    
                    # 确保所有必要字段都存在
                    for field in ['amount', 'price', 'weight', 'count_day']:
                        if field not in stock_data:
                            stock_data[field] = 0
                    
                    daily_holdings_list.append(stock_data)
                    stock_count += 1
                    total_position_value += position_value
            
            # 更新账户汇总信息
            daily_account_data['stock_count'] = stock_count
            daily_account_data['total_position_value'] = total_position_value
            
            # 计算比例
            if account_value > 0:
                daily_account_data['cash_ratio'] = cash / account_value
                daily_account_data['position_ratio'] = total_position_value / account_value
            
            daily_account_list.append(daily_account_data)
            
            print(f"日期 {date_str}: 现金={cash:.2f}, 账户价值={account_value:.2f}, 持股数={stock_count}, 持仓价值={total_position_value:.2f}")
            
        except Exception as e:
            print(f"处理日期 {date_str} 时出错: {e}")
            continue
    
    # 创建两个DataFrame
    if daily_account_list:
        # 表1：每日账户汇总
        account_df = pd.DataFrame(daily_account_list)
        
        # 确保数值列的数据类型正确
        numeric_columns = ['cash', 'account_value', 'stock_count', 'total_position_value', 'cash_ratio', 'position_ratio']
        for col in numeric_columns:
            if col in account_df.columns:
                account_df[col] = pd.to_numeric(account_df[col], errors='coerce').fillna(0)
        
        # 保存账户汇总表
        account_file = f'{output_dir}/{factor_name_pathlib}_每日账户汇总.csv'
        account_df.to_csv(account_file, index=False, encoding='utf-8-sig')
        print(f"每日账户汇总表已保存到: {account_file}")
        
        print("=" * 60)
        print("表1：每日账户汇总")
        print("=" * 60)
        print(f"交易日数: {len(account_df)}")
        print(f"账户价值范围: {account_df['account_value'].min():.2f} - {account_df['account_value'].max():.2f}")
        print(f"现金范围: {account_df['cash'].min():.2f} - {account_df['cash'].max():.2f}")
        print(f"平均持股数: {account_df['stock_count'].mean():.1f}")
        print(f"平均现金比例: {account_df['cash_ratio'].mean():.2%}")
        print(f"平均持仓比例: {account_df['position_ratio'].mean():.2%}")
        
    else:
        account_df = pd.DataFrame()
        print("没有账户汇总数据")
    
    if daily_holdings_list:
        # 表2：每日股票持仓
        holdings_df = pd.DataFrame(daily_holdings_list)
        
        # 确保数值列的数据类型正确
        numeric_columns = ['amount', 'price', 'weight', 'count_day', 'position_value']
        for col in numeric_columns:
            if col in holdings_df.columns:
                holdings_df[col] = pd.to_numeric(holdings_df[col], errors='coerce').fillna(0)
        
        # 保存股票持仓表
        holdings_file = f'{output_dir}/{factor_name_pathlib}_每日股票持仓.csv'
        holdings_df.to_csv(holdings_file, index=False, encoding='utf-8-sig')
        print(f"每日股票持仓表已保存到: {holdings_file}")
        
        print("\n" + "=" * 60)
        print("表2：每日股票持仓")
        print("=" * 60)
        print(f"股票持仓记录数: {len(holdings_df)}")
        print(f"涉及股票数: {holdings_df['stock_code'].nunique()}")
        print(f"平均每日持股数: {holdings_df.groupby('date')['stock_code'].count().mean():.1f}")
        
        # 最常持有的股票
        top_stocks = holdings_df['stock_code'].value_counts().head(10)
        print("最常持有的前10只股票:")
        for stock, count in top_stocks.items():
            print(f"  {stock}: {count} 天")
            
    else:
        holdings_df = pd.DataFrame()
        print("没有股票持仓数据")
    
    return account_df, holdings_df

# 使用改进的函数处理持仓数据，分成两张表
account_df, holdings_df = process_holdings_data_separated(positions_dict)



# 可视化部分

In [ ]:
# 可视化分析代码
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.patches import Rectangle
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.offline as pyo
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 设置字体家族为支持中文的字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows 系统常用黑体
# plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  # Windows 微软雅黑
# plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # macOS 常用
# plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']  # Linux 常用
# 解决负号显示问题
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# 1. 账户价值趋势分析函数 - 改进版本（中文标签）
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import numpy as np

def plot_account_value_analysis_improved(account_df, save_path=f'logs/{factor_name_pathlib}'):
    """
    绘制账户价值趋势分析图表
    包括：净值曲线、回撤分析、滚动收益率和波动率分析
    """
    if account_df.empty:
        print("❌ 账户数据为空，无法绘制图表")
        return
    
    # 确保日期列为datetime类型
    account_df = account_df.copy()
    account_df['date'] = pd.to_datetime(account_df['date'])
    account_df = account_df.sort_values('date')
    
    # 计算净值（相对于初始值）
    initial_value = account_df['account_value'].iloc[0]
    account_df['net_value'] = account_df['account_value'] / initial_value
    
    # 计算回撤
    account_df['cummax'] = account_df['net_value'].cummax()
    account_df['drawdown'] = (account_df['net_value'] - account_df['cummax']) / account_df['cummax']
    
    # 计算日收益率
    account_df['daily_return'] = account_df['net_value'].pct_change()
    
    # 计算滚动指标（30天窗口）
    window = 30
    account_df['rolling_return'] = account_df['daily_return'].rolling(window=window).mean() * 252  # 年化
    account_df['rolling_volatility'] = account_df['daily_return'].rolling(window=window).std() * np.sqrt(252)  # 年化
    account_df['rolling_sharpe'] = account_df['rolling_return'] / account_df['rolling_volatility']
    
    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('账户价值趋势分析 - 改进版本', fontsize=16, fontweight='bold')
    
    # 1. 净值曲线
    ax1 = axes[0, 0]
    ax1.plot(account_df['date'], account_df['net_value'], linewidth=2, color='#2E86AB', label='净值')
    ax1.axhline(y=1, color='red', linestyle='--', alpha=0.7, label='初始价值')
    ax1.set_title('净值曲线', fontsize=14, fontweight='bold')
    ax1.set_ylabel('净值')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. 回撤分析
    ax2 = axes[0, 1]
    ax2.fill_between(account_df['date'], account_df['drawdown'], 0, 
                     color='red', alpha=0.3, label='回撤')
    ax2.plot(account_df['date'], account_df['drawdown'], color='red', linewidth=1)
    ax2.set_title('回撤分析', fontsize=14, fontweight='bold')
    ax2.set_ylabel('回撤比例')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 滚动年化收益率（30天窗口）
    ax3 = axes[1, 0]
    ax3.plot(account_df['date'], account_df['rolling_return'] * 100, 
             linewidth=2, color='#28A745', label=f'{window}日滚动收益率')
    ax3.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    ax3.set_title(f'滚动年化收益率（{window}日窗口）', fontsize=14, fontweight='bold')
    ax3.set_ylabel('年化收益率 (%)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. 滚动波动率和夏普比率
    ax4 = axes[1, 1]
    ax4_twin = ax4.twinx()
    
    # 绘制波动率
    line1 = ax4.plot(account_df['date'], account_df['rolling_volatility'] * 100, 
                     linewidth=2, color='#FF6B6B', label=f'{window}日滚动波动率')
    ax4.set_ylabel('年化波动率 (%)', color='#FF6B6B')
    ax4.tick_params(axis='y', labelcolor='#FF6B6B')
    
    # 在次y轴绘制夏普比率
    line2 = ax4_twin.plot(account_df['date'], account_df['rolling_sharpe'], 
                          linewidth=2, color='#4ECDC4', label=f'{window}日滚动夏普比率')
    ax4_twin.set_ylabel('夏普比率', color='#4ECDC4')
    ax4_twin.tick_params(axis='y', labelcolor='#4ECDC4')
    ax4_twin.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    
    ax4.set_title(f'风险指标（{window}日窗口）', fontsize=14, fontweight='bold')
    ax4.set_xlabel('日期')
    
    # 合并图例
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax4.legend(lines, labels, loc='upper left')
    ax4.grid(True, alpha=0.3)
    
    # 设置x轴日期格式
    for ax in axes.flat:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=8)
    
    plt.tight_layout()
    
    # 保存图表
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    filename = f'{save_path}/{factor_name_pathlib}_账户价值分析.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"📊 改进版账户价值分析图表已保存: {filename}")
    
    plt.show()
    
    # 打印关键统计信息
    print("\n" + "="*60)
    print("📈 关键账户价值指标")
    print("="*60)
    print(f"初始账户价值: {initial_value:,.2f}")
    print(f"最终账户价值: {account_df['account_value'].iloc[-1]:,.2f}")
    print(f"总收益率: {(account_df['net_value'].iloc[-1] - 1) * 100:.2f}%")
    print(f"最大回撤: {account_df['drawdown'].min() * 100:.2f}%")
    print(f"平均现金比例: {account_df['cash_ratio'].mean() * 100:.2f}%")
    print(f"平均持仓比例: {account_df['position_ratio'].mean() * 100:.2f}%")
    
    # 额外风险指标
    print(f"\n📊 风险指标:")
    print(f"年化收益率: {account_df['daily_return'].mean() * 252 * 100:.2f}%")
    print(f"年化波动率: {account_df['daily_return'].std() * np.sqrt(252) * 100:.2f}%")
    print(f"夏普比率: {(account_df['daily_return'].mean() * 252) / (account_df['daily_return'].std() * np.sqrt(252)):.2f}")
    print(f"最大日亏损: {account_df['daily_return'].min() * 100:.2f}%")
    print(f"最大日收益: {account_df['daily_return'].max() * 100:.2f}%")
    
    # 滚动指标汇总
    print(f"\n📈 滚动指标（30日窗口）:")
    print(f"平均滚动收益率: {account_df['rolling_return'].mean() * 100:.2f}%")
    print(f"平均滚动波动率: {account_df['rolling_volatility'].mean() * 100:.2f}%")
    print(f"平均滚动夏普比率: {account_df['rolling_sharpe'].mean():.2f}")
    print(f"最大滚动夏普比率: {account_df['rolling_sharpe'].max():.2f}")
    print(f"最小滚动夏普比率: {account_df['rolling_sharpe'].min():.2f}")
    
    return fig

# 调用函数绘制改进版账户价值分析图表
if not account_df.empty:
    plot_account_value_analysis_improved(account_df)
else:
    print("❌ 账户数据为空，请先运行持仓数据处理代码")


In [ ]:
# 风险分析函数（中文标签）
def plot_risk_analysis_fixed(account_df, save_path=f'logs/{factor_name_pathlib}'):
    """
    绘制风险分析图表（中文标签）
    包括：日收益率分布、滚动波动率、夏普比率、VaR分析
    """
    if account_df.empty:
        print("❌ 账户数据为空，无法绘制图表")
        return
    
    # 确保日期列为datetime类型并排序
    account_df = account_df.copy()
    account_df['date'] = pd.to_datetime(account_df['date'])
    account_df = account_df.sort_values('date')
    
    # 计算日收益率
    account_df['daily_return'] = account_df['account_value'].pct_change()
    account_df['daily_return_pct'] = account_df['daily_return'] * 100
    
    # 计算滚动波动率（30天窗口）
    account_df['rolling_volatility'] = account_df['daily_return'].rolling(window=30).std() * np.sqrt(252) * 100
    
    # 计算滚动夏普比率（30天窗口，假设无风险利率为3%）
    risk_free_rate = 0.03
    account_df['rolling_sharpe'] = (account_df['daily_return'].rolling(window=30).mean() * 252 - risk_free_rate) / (account_df['daily_return'].rolling(window=30).std() * np.sqrt(252))
    
    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('风险分析', fontsize=16, fontweight='bold')
    
    # 1. 日收益率分布
    ax1 = axes[0, 0]
    returns_data = account_df['daily_return_pct'].dropna()
    ax1.hist(returns_data, bins=50, alpha=0.7, color='#2E86AB', edgecolor='black', density=True)
    ax1.axvline(returns_data.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {returns_data.mean():.2f}%')
    ax1.axvline(returns_data.median(), color='orange', linestyle='--', linewidth=2, label=f'中位数: {returns_data.median():.2f}%')
    ax1.set_title('日收益率分布', fontsize=14, fontweight='bold')
    ax1.set_xlabel('日收益率 (%)')
    ax1.set_ylabel('密度')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. 滚动波动率
    ax2 = axes[0, 1]
    ax2.plot(account_df['date'], account_df['rolling_volatility'], 
             linewidth=2, color='#A23B72', label='30日滚动波动率')
    ax2.axhline(y=account_df['rolling_volatility'].mean(), color='red', 
                linestyle='--', alpha=0.7, label=f'平均波动率: {account_df["rolling_volatility"].mean():.2f}%')
    ax2.set_title('滚动波动率', fontsize=14, fontweight='bold')
    ax2.set_ylabel('年化波动率 (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 滚动夏普比率
    ax3 = axes[1, 0]
    ax3.plot(account_df['date'], account_df['rolling_sharpe'], 
             linewidth=2, color='#F18F01', label='30日滚动夏普比率')
    ax3.axhline(y=account_df['rolling_sharpe'].mean(), color='red', 
                linestyle='--', alpha=0.7, label=f'平均夏普比率: {account_df["rolling_sharpe"].mean():.2f}')
    ax3.axhline(y=1, color='green', linestyle='--', alpha=0.7, label='夏普比率=1')
    ax3.set_title('滚动夏普比率', fontsize=14, fontweight='bold')
    ax3.set_ylabel('夏普比率')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. VaR分析（风险价值）
    ax4 = axes[1, 1]
    # 计算不同置信水平的VaR
    confidence_levels = [0.95, 0.99]
    var_values = []
    var_labels = []
    
    for conf in confidence_levels:
        var_value = np.percentile(returns_data, (1 - conf) * 100)
        var_values.append(var_value)
        var_labels.append(f'{conf*100}% VaR: {var_value:.2f}%')
    
    # 绘制VaR线
    ax4.hist(returns_data, bins=50, alpha=0.7, color='#2E86AB', edgecolor='black', density=True)
    for i, (var_val, var_label) in enumerate(zip(var_values, var_labels)):
        ax4.axvline(var_val, color=['red', 'darkred'][i], linestyle='--', 
                   linewidth=2, label=var_label)
    
    ax4.set_title('VaR分析', fontsize=14, fontweight='bold')
    ax4.set_xlabel('日收益率 (%)')
    ax4.set_ylabel('密度')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 设置x轴日期格式
    for ax in [axes[0, 1], axes[1, 0]]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=8)
    
    plt.tight_layout()
    
    # 保存图表
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    filename = f'{save_path}/{factor_name_pathlib}_风险分析.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"📊 风险分析图表已保存: {filename}")
    
    plt.show()
    
    # 打印关键风险指标
    print("\n" + "="*60)
    print("⚠️ 关键风险分析指标")
    print("="*60)
    print(f"平均日收益率: {returns_data.mean():.4f}%")
    print(f"日收益率标准差: {returns_data.std():.4f}%")
    print(f"年化波动率: {returns_data.std() * np.sqrt(252):.2f}%")
    print(f"夏普比率: {(returns_data.mean() * 252 - risk_free_rate) / (returns_data.std() * np.sqrt(252)):.2f}")
    print(f"最大日亏损: {returns_data.min():.2f}%")
    print(f"最大日收益: {returns_data.max():.2f}%")
    print(f"95% VaR: {var_values[0]:.2f}%")
    print(f"99% VaR: {var_values[1]:.2f}%")
    
    # 计算偏度和峰度
    from scipy import stats
    skewness = stats.skew(returns_data)
    kurtosis = stats.kurtosis(returns_data)
    print(f"偏度: {skewness:.2f}")  
    print(f"峰度: {kurtosis:.2f}")
    
    return fig

# 调用风险分析函数
if not account_df.empty:
    plot_risk_analysis_fixed(account_df)
else:
    print("❌ 账户数据为空，请先运行持仓数据处理代码")


In [ ]:
# 收益分析函数（中文标签）
def plot_returns_analysis_english(report_df, account_df, save_path=f'logs/{factor_name_pathlib}'):
    """
    绘制收益分析图表（中文标签）
    包括：累计收益率曲线、年化收益率、月度收益率热力图、与基准对比
    """
    if account_df.empty:
        print("❌ 账户数据为空，无法绘制图表")
        return
    
    # 确保日期列为datetime类型并排序
    account_df = account_df.copy()
    account_df['date'] = pd.to_datetime(account_df['date'])
    account_df = account_df.sort_values('date')
    
    # 计算日收益率和累计收益率
    account_df['daily_return'] = account_df['account_value'].pct_change()
    account_df['cumulative_return'] = (1 + account_df['daily_return']).cumprod() - 1
    account_df['cumulative_return_pct'] = account_df['cumulative_return'] * 100
    
    # 准备基准数据：将report_df的索引转换为日期格式进行匹配
    report_df_copy = report_df.copy()
    report_df_copy['date'] = pd.to_datetime(report_df_copy.index)
    
    # 将基准数据合并到account_df中
    account_df = account_df.merge(
        report_df_copy[['date', 'bench', 'bench_cumulative_return']], 
        on='date', 
        how='left'
    )
    
    # 计算基准的百分比形式
    account_df['bench_cumulative_return_pct'] = account_df['bench_cumulative_return'] * 100
    account_df['bench_daily_return_pct'] = account_df['bench'] * 100
    
    # 计算年化收益率
    total_days = (account_df['date'].iloc[-1] - account_df['date'].iloc[0]).days
    years = total_days / 365.25
    annualized_return = (account_df['account_value'].iloc[-1] / account_df['account_value'].iloc[0]) ** (1/years) - 1
    
    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('收益分析', fontsize=16, fontweight='bold')
    
    # 1. 累计收益率曲线
    ax1 = axes[0, 0]
    ax1.plot(account_df['date'], account_df['cumulative_return_pct'], 
             linewidth=2, color='#2E86AB', label='策略累计收益率')
    # 添加基准累计收益率曲线
    ax1.plot(account_df['date'], account_df['bench_cumulative_return_pct'], 
             linewidth=2, color='#FF6B35', label='基准累计收益率')
    ax1.axhline(y=0, color='red', linestyle='--', alpha=0.7, label='零收益线')
    ax1.set_title('累计收益率曲线 vs 基准', fontsize=14, fontweight='bold')
    ax1.set_ylabel('累计收益率 (%)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. 日收益率时间序列
    ax2 = axes[0, 1]
    ax2.plot(account_df['date'], account_df['daily_return'] * 100, 
             linewidth=1, color='#A23B72', alpha=0.7, label='策略日收益率')
    # # 添加基准日收益率曲线
    # ax2.plot(account_df['date'], account_df['bench_daily_return_pct'], 
    #          linewidth=1, color='#FF6B35', alpha=0.1, label='基准日收益率')
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7)
    ax2.set_title('日收益率时间序列 vs 基准', fontsize=14, fontweight='bold')
    ax2.set_ylabel('日收益率 (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 月度收益率热力图
    ax3 = axes[1, 0]
    # 创建月度收益率数据
    account_df['year'] = account_df['date'].dt.year
    account_df['month'] = account_df['date'].dt.month
    
    monthly_returns = account_df.groupby(['year', 'month'])['daily_return'].apply(lambda x: (1 + x).prod() - 1) * 100
    monthly_returns = monthly_returns.reset_index()
    monthly_returns = monthly_returns.pivot(index='year', columns='month', values='daily_return')
    
    if not monthly_returns.empty:
        sns.heatmap(monthly_returns, annot=True, fmt='.1f', cmap='RdYlGn', 
                   center=0, ax=ax3, cbar_kws={'label': '月度收益率 (%)'})
        ax3.set_title('月度收益率热力图', fontsize=14, fontweight='bold')
        ax3.set_xlabel('月份')
        ax3.set_ylabel('年份')
    else:
        ax3.text(0.5, 0.5, '无月度数据', ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title('月度收益率热力图', fontsize=14, fontweight='bold')
    
    # 4. 年度收益率分布箱线图
    ax4 = axes[1, 1]
    yearly_returns = []
    yearly_labels = []
    
    for year in sorted(account_df['year'].unique()):
        year_data = account_df[account_df['year'] == year]['daily_return'] * 100
        if not year_data.empty:
            yearly_returns.append(year_data.dropna())
            yearly_labels.append(str(year))
    
    if yearly_returns:
        ax4.boxplot(yearly_returns, labels=yearly_labels)
        ax4.set_title('年度收益率分布箱线图', fontsize=14, fontweight='bold')
        ax4.set_xlabel('年份')
        ax4.set_ylabel('日收益率 (%)')
        ax4.grid(True, alpha=0.3)
    else:
        ax4.text(0.5, 0.5, '无年度数据', ha='center', va='center', transform=ax4.transAxes)
        ax4.set_title('年度收益率分布箱线图', fontsize=14, fontweight='bold')
    
    # 设置x轴日期格式
    for ax in [axes[0, 0], axes[0, 1]]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=8)
    
    plt.tight_layout()
    
    # 保存图表
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    filename = f'{save_path}/{factor_name_pathlib}_收益分析.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"📊 收益分析图表已保存: {filename}")
    
    plt.show()
    
    # 打印关键收益指标
    print("\n" + "="*60)
    print("💰 关键收益分析指标")
    print("="*60)
    print(f"总收益率: {account_df['cumulative_return_pct'].iloc[-1]:.2f}%")
    print(f"年化收益率: {annualized_return * 100:.2f}%")
    print(f"最大日收益: {account_df['daily_return'].max() * 100:.2f}%")
    print(f"最大日亏损: {account_df['daily_return'].min() * 100:.2f}%")
    print(f"平均日收益率: {account_df['daily_return'].mean() * 100:.4f}%")
    print(f"收益率标准差: {account_df['daily_return'].std() * 100:.4f}%")
    
    # 计算胜率
    positive_days = (account_df['daily_return'] > 0).sum()
    total_days = len(account_df['daily_return'].dropna())
    win_rate = positive_days / total_days if total_days > 0 else 0
    print(f"胜率: {win_rate * 100:.2f}%")
    
    # 计算最大连续亏损天数
    daily_returns = account_df['daily_return'].dropna()
    consecutive_losses = 0
    max_consecutive_losses = 0
    for ret in daily_returns:
        if ret < 0:
            consecutive_losses += 1
            max_consecutive_losses = max(max_consecutive_losses, consecutive_losses)
        else:
            consecutive_losses = 0
    
    print(f"最大连续亏损天数: {max_consecutive_losses}")
    
    return fig

# 调用收益分析函数
if not account_df.empty and not report_df.empty:
    plot_returns_analysis_english(report_df,account_df)
else:
    print("❌ 账户数据为空，请先运行持仓数据处理代码")
